# End-to-End Machine Learning Project: Predicting Medical Insurance Costs

In this notebook, we will to build a regression model that predicts medical insurance charges based on personal attributes.

## 1. Introduction
We will use the **Medical Cost Personal Datasets**. It contains features like age, sex, BMI, number of children, smoking status, and region to predict the insurance `charges`.
- **Problem:** Supervised Learning (Regression)
- **Why suitable?** It is tabular, contains a mix of categorical and numerical features, and requires clear preprocessing (scaling, encoding) before training models like Linear Regression or Random Forests. It is intuitive and accessible.

### 1.1. Install Dependencies
Install necessary libraries if you are running this notebook for the first time


In [ ]:
%pip install -q pip pandas numpy matplotlib seaborn scikit-learn jinja2

print("✅ Installation check complete. Ready to proceed!")

### 1.2. Import Libraries

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

# Make this notebook's output stable across runs
np.random.seed(42)

# Plot formatting
plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)
sns.set_theme(style='whitegrid')

print("✅ Libraries imported successfully!")
print(f"🔹 Python: {sys.version.split(' ')[0]}")
print(f"🔹 Pandas: {pd.__version__}")
print(f"🔹 NumPy: {np.__version__}")
print(f"🔹 Scikit-Learn: {sklearn.__version__}")
print(f"🔹 Seaborn: {sns.__version__}")

## 2. Data Loading
We can download the dataset directly from a public GitHub repository hosting Kaggle datasets.

In [ ]:
url = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'
insurance = pd.read_csv(url)

# We show a sample of the first 5 and last 5 rows
sample_df = pd.concat([insurance.head(5), insurance.tail(5)])
display(sample_df.style.set_caption("🩺 Medical Insurance Dataset (First 5 & Last 5 Rows)") \
        .set_table_styles([{'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold')]}]) \
        .background_gradient(subset=['charges', 'bmi'], cmap='Blues'))

## 3. Exploratory Data Analysis (EDA)
Understanding our features and target variable (`charges`).

In [ ]:
# Summary statistics for numerical attributes with a visual gradient!
display(insurance.describe().T.style.set_caption("📊 Summary Statistics") \
        .set_table_styles([{'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold')]}]) \
        .background_gradient(cmap='viridis', axis=1))

In [ ]:
# A more visual interpretation instead of standard insurance.info()
info_df = pd.DataFrame({
    'Data Type': insurance.dtypes,
    'Non-Null Count': insurance.notnull().sum(),
    'Null Count': insurance.isnull().sum(),
    'Unique Values': insurance.nunique()
})

display(info_df.style.set_caption("📋 Dataset Information & Missing Values") \
        .set_table_styles([{'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold')]}]) \
        .background_gradient(subset=['Null Count'], cmap='Reds') \
        .background_gradient(subset=['Unique Values'], cmap='Greens'))

> *Insight:* There are 1338 instances and 0 missing values. Great! The categorical features are `sex`, `smoker`, and `region`.

In [ ]:
# Visualizing distributions
# We use Seaborn's histplot with a colorful palette and Kernel Density Estimate (KDE) lines for a modern look
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

numeric_cols = insurance.select_dtypes(include=[np.number]).columns
colors = sns.color_palette("husl", len(numeric_cols))

for i, col in enumerate(numeric_cols):
    sns.histplot(insurance[col], kde=True, ax=axes[i], color=colors[i], bins=30)
    axes[i].set_title(f"Distribution of {col}", fontsize=14, fontweight='bold')
    axes[i].set_xlabel(col.capitalize(), fontsize=12)
    axes[i].set_ylabel('Count', fontsize=12)
    
plt.tight_layout()
plt.show()

> *Insight:* `charges` is heavily right-skewed. Most people have lower medical costs, but a tail exists for very high costs. `bmi` looks normally distributed.

In [ ]:
# Creating a scatterplot to see relationships with the target using sns.lmplot for a modern look
sns.lmplot(
    data=insurance, 
    x='bmi', 
    y='charges', 
    hue='smoker', 
    palette='Set1',
    aspect=1.5, 
    height=6,
    scatter_kws={'alpha':0.6, 'edgecolor': 'w'}
)

plt.title('BMI vs Charges, colored by Smoking Status', fontsize=16, fontweight='bold', pad=15)
plt.show()

> *Insight:* Smoking clearly separates the data! Smokers with high BMI have exponentially higher medical charges. This indicates `smoker` is a crucial feature.

Let's also look at the correlation matrix.

In [ ]:
# We'll encode 'smoker' temporarily just to see numerical correlation
corr_df = insurance.copy()
corr_df['smoker'] = corr_df['smoker'].map({'yes': 1, 'no': 0})
corr_mat = corr_df[['age', 'bmi', 'children', 'smoker', 'charges']].corr()

# Create a mask to hide the upper triangle for a cleaner look
mask = np.triu(np.ones_like(corr_mat, dtype=bool))

plt.figure(figsize=(8, 6))
sns.heatmap(corr_mat, mask=mask, annot=True, cmap='RdYlBu_r', 
            fmt=".2f", linewidths=1.5, cbar_kws={"shrink": .8})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=15)
plt.show()

## 4. Data Preprocessing
Let's split the data, and build a Scikit-Learn Pipeline to handle scaling and encoding.

In [ ]:
from sklearn.model_selection import train_test_split

X = insurance.drop('charges', axis=1)
y = insurance['charges']

# 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("✂️ Dataset successfully split!")
print(f"Training Set: {X_train.shape[0]} rows")
print(f"Test Set:     {X_test.shape[0]} rows")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn import set_config

num_attribs = ['age', 'bmi', 'children']
cat_attribs = ['sex', 'smoker', 'region']

num_pipeline = Pipeline([
    ('std_scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessing = ColumnTransformer([
    ('num', num_pipeline, num_attribs),
    ('cat', cat_pipeline, cat_attribs),
])

X_train_prepared = preprocessing.fit_transform(X_train)
X_test_prepared = preprocessing.transform(X_test)

# Display interactive Pipeline architecture!
set_config(display='diagram')
preprocessing

## 5. Modeling
We'll try **Linear Regression**, **Decision Tree**, and **Random Forest**.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

lin_reg = LinearRegression()
lin_reg.fit(X_train_prepared, y_train)

tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(X_train_prepared, y_train)

forest_reg = RandomForestRegressor(random_state=42)
forest_reg.fit(X_train_prepared, y_train)

## 6. Model Evaluation
Let's use Root Mean Squared Error (RMSE) and Cross-Validation to evaluate.

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score

def get_scores(model, X, y):
    scores = cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=10)
    rmse_scores = np.sqrt(-scores)
    return {
        "Model": model.__class__.__name__,
        "Mean RMSE": rmse_scores.mean(),
        "Std Dev": rmse_scores.std()
    }

# Gather findings
results = [
    get_scores(lin_reg, X_train_prepared, y_train),
    get_scores(tree_reg, X_train_prepared, y_train),
    get_scores(forest_reg, X_train_prepared, y_train)
]

# Display as a formatted, interactive dataframe
results_df = pd.DataFrame(results).set_index("Model")
display(results_df.style.set_caption("🏆 Cross-Validation Model Comparison") \
        .set_table_styles([{'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold')]}]) \
        .background_gradient(subset=['Mean RMSE'], cmap='RdYlGn_r') \
        .format("{:.2f}"))

> *Insight:* Random Forest performs the best overall (lowest Mean RMSE). The Decision Tree is overfitting heavily (worse validation score than Random Forest). Linear Regression is decent but Random Forest captures nonlinear interactions (like the smoking/BMI relationship seen in EDA).

## 7. Hyperparameter Tuning
Let's fine-tune our Random Forest using `GridSearchCV`.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = [
    {'n_estimators': [30, 50, 100], 'max_features': [2, 4, 6, 8]},
    {'bootstrap': [False], 'n_estimators': [3, 10], 'max_features': [2, 3, 4]}
]

forest_reg_tune = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(forest_reg_tune, param_grid, cv=5, scoring='neg_mean_squared_error', return_train_score=True)
grid_search.fit(X_train_prepared, y_train)

best_model = grid_search.best_estimator_

# Prettify the output
params_df = pd.DataFrame([grid_search.best_params_])
display(params_df.style.set_caption("⚙️ Optimal Random Forest Hyperparameters") \
        .set_table_styles([{'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold')]}]) \
        .set_properties(**{'background-color': '#f4f4f9', 'color': 'black', 'border-color': 'white'}))

## 8. Final Model & Interpretation
We evaluate our final selected and tuned model on the Test Set.

In [ ]:
# Use native np.sqrt to avoid deprecation issues in sklearn 1.4+ with mean_squared_error
final_predictions = best_model.predict(X_test_prepared)
final_mse = mean_squared_error(y_test, final_predictions)
final_rmse = np.sqrt(final_mse)
print(f"🎯 Final Test RMSE: {final_rmse:.2f}")

In [ ]:
# Feature Importances
feature_importances = best_model.feature_importances_

# Get feature names from the pipeline
cat_encoder = preprocessing.named_transformers_['cat'].named_steps['encoder']
cat_one_hot_attribs = list(cat_encoder.get_feature_names_out(cat_attribs))
attributes = num_attribs + cat_one_hot_attribs

sorted_importances = sorted(zip(feature_importances, attributes), reverse=True)

# Plotting
plt.figure(figsize=(10, 6))
# Using hue along with legend=False suppresses the recent Seaborn future warning
sns.barplot(
    x=[x[0] for x in sorted_importances], 
    y=[x[1] for x in sorted_importances], 
    hue=[x[1] for x in sorted_importances],
    palette='viridis',
    legend=False
)
plt.title('Feature Importances (What drives medical costs?)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Relative Importance', fontsize=14)
plt.ylabel('Features', fontsize=14)
plt.show()

> *Insight:* `smoker_yes` is overwhelmingly the most important feature, followed by `bmi` and `age`. This aligns perfectly with our earlier EDA!

## 9. Conclusion
- **Summary:** We successfully explored the Medical Insurance dataset and built a Random Forest Regressor to predict charges. Over 60% of the model's predictive power comes from knowing whether a patient smokes.
- **Limitations:** The model may struggle with extreme outliers (exceedingly rare medical conditions not represented in generic features). The dataset is relatively small, which limits the diversity of cases.
- **Extensions:** We could explore feature engineering (e.g., categorizing BMI into bins like "Obese" vs "Normal" and creating interactions with 'smoker'.)